# Phase Boundary Training (Google Colab)

This notebook is designed for the `Phase Boundary Testing` workflow in this repository. It will:

- install the Python dependencies needed in Colab
- mount Google Drive
- clone the GitHub repository into the Colab runtime
- train both DEC and IDEC on the available phase-boundary data sources
- save the run artifacts back into the repo checkout and copy them to Drive
- optionally commit and push the new results to GitHub

Sensitive values such as GitHub username, email, and token are requested at runtime and are not hardcoded in the notebook.

## Colab Setup

Run the cells in order. The training cell uses the existing `run_pipeline(...)` entry point from `Phase Boundary Testing/src/train_phase_boundary_models.py`, so the notebook stays aligned with the repo code.

In [ ]:
import importlib
import subprocess
import sys

REQUIRED_PACKAGES = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "sklearn": "scikit-learn",
    "h5py": "h5py",
    "tqdm": "tqdm",
    "hyperspy": "hyperspy",
    "ncempy": "ncempy",
    "tensorflow": "tensorflow",
}

for module_name, package_name in REQUIRED_PACKAGES.items():
    try:
        importlib.import_module(module_name)
    except ImportError:
        print(f"Installing {package_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])

print("Environment ready.")

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
from datetime import datetime
from pathlib import Path
import getpass
import json
import os
import shutil
import subprocess
import sys

COLAB_ROOT = Path("/content")
DEFAULT_REPO_DIRNAME = "cmu-vpcf-project"
DEFAULT_DRIVE_RESULTS_ROOT = Path("/content/drive/MyDrive/CMU_vPCF_Project/PhaseBoundaryResults")


def prompt_text(label, default=None, secret=False, allow_blank=False):
    suffix = f" [{default}]" if default else ""
    prompt = f"{label}{suffix}: "
    value = getpass.getpass(prompt) if secret else input(prompt)
    value = value.strip()
    if not value and default is not None:
        value = str(default)
    if not value and not allow_blank:
        raise ValueError(f"{label} is required.")
    return value


def write_git_askpass(username: str, token: str) -> Path:
    askpass_path = COLAB_ROOT / "git_askpass.py"
    askpass_path.write_text(
        "import os, sys\n"
        "prompt = sys.argv[1].lower() if len(sys.argv) > 1 else ''\n"
        "if 'username' in prompt:\n"
        "    print(os.environ.get('GITHUB_USERNAME', ''))\n"
        "else:\n"
        "    print(os.environ.get('GITHUB_TOKEN', ''))\n",
        encoding="utf-8",
    )
    return askpass_path


def run_git(args, cwd=None, username=None, token=None):
    env = os.environ.copy()
    if username is not None and token is not None:
        askpass_path = write_git_askpass(username, token)
        env["GIT_ASKPASS"] = str(askpass_path)
        env["GIT_TERMINAL_PROMPT"] = "0"
        env["GITHUB_USERNAME"] = username
        env["GITHUB_TOKEN"] = token
    print("git", " ".join(args))
    subprocess.run(["git", *args], cwd=cwd, check=True, env=env)


repo_url = prompt_text("GitHub repo HTTPS URL", default="https://github.com/<owner>/<repo>.git")
repo_dir_name = prompt_text("Local repo folder name in Colab", default=DEFAULT_REPO_DIRNAME)
reuse_existing = prompt_text("Reuse an existing /content checkout if present? (y/n)", default="n").lower().startswith("y")
drive_results_root = Path(prompt_text("Google Drive results folder", default=str(DEFAULT_DRIVE_RESULTS_ROOT)))
drive_results_root.mkdir(parents=True, exist_ok=True)

repo_root = COLAB_ROOT / repo_dir_name
if repo_root.exists() and not reuse_existing:
    shutil.rmtree(repo_root)

if not repo_root.exists():
    clone_private = prompt_text("Does cloning require GitHub credentials? (y/n)", default="n").lower().startswith("y")
    if clone_private:
        clone_username = prompt_text("GitHub username")
        clone_token = prompt_text("GitHub personal access token", secret=True)
        run_git(["clone", repo_url, str(repo_root)], username=clone_username, token=clone_token)
    else:
        run_git(["clone", repo_url, str(repo_root)])
else:
    print(f"Reusing existing repo at {repo_root}")

phase_boundary_dir = repo_root / "Phase Boundary Testing"
src_dir = phase_boundary_dir / "src"
data_dir = phase_boundary_dir / "Data"
results_dir = phase_boundary_dir / "Results"

if not phase_boundary_dir.exists():
    raise FileNotFoundError(f"Could not find {phase_boundary_dir}")

if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from train_phase_boundary_models import run_pipeline

print(f"Repo root: {repo_root}")
print(f"Phase Boundary dir: {phase_boundary_dir}")
print(f"Drive results root: {drive_results_root}")

In [ ]:
H5_FILE = data_dir / "SyntheticModel_HfO_80pm_vPCFs_65.h5"
DM3_FILE = data_dir / "SyntheticModel_HfO_80pm_gaussian_HAADF.dm3"

for required_path in [H5_FILE, DM3_FILE]:
    print(required_path, "exists:" , required_path.exists())

RUN_TAG = input("Run tag [colab_full_train]: ").strip() or "colab_full_train"
TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_ROOT = results_dir / f"{RUN_TAG}_{TIMESTAMP}"
DRIVE_RUN_ROOT = drive_results_root / f"{RUN_TAG}_{TIMESTAMP}"
RUN_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_RUN_ROOT.mkdir(parents=True, exist_ok=True)

TRAINING_KWARGS = {
    "model": "both",
    "n_clusters": 3,
    "feature_method": "flatten",
    "normalize": "minmax",
    "downsample_factor": None,
    "max_frames": None,
    "hidden_dims": [500, 500, 2000],
    "pretrain_epochs": 50,
    "maxiter": 8000,
    "update_interval": 140,
    "batch_size": 256,
    "gamma": 0.1,
    "verbose": True,
}

RUN_CONFIGS = [
    {
        "name": "h5_only",
        "h5_file": str(H5_FILE),
        "dm3_file": None,
    },
    {
        "name": "dm3_only",
        "h5_file": None,
        "dm3_file": str(DM3_FILE),
    },
    {
        "name": "combined",
        "h5_file": str(H5_FILE),
        "dm3_file": str(DM3_FILE),
    },
]

print("Repo results directory:", RUN_ROOT)
print("Drive results directory:", DRIVE_RUN_ROOT)
print("If you want a lighter smoke test first, edit TRAINING_KWARGS['max_frames'] before running the next cell.")

In [ ]:
import pandas as pd

run_summaries = []

for config in RUN_CONFIGS:
    output_dir = RUN_ROOT / config["name"]
    print("\n" + "=" * 80)
    print(f"Starting run: {config['name']}")
    print("=" * 80)

    results = run_pipeline(
        h5_file=config["h5_file"],
        dm3_file=config["dm3_file"],
        output_dir=str(output_dir),
        **TRAINING_KWARGS,
    )

    summary_path = Path(results["summary_path"])
    drive_output_dir = DRIVE_RUN_ROOT / config["name"]
    shutil.copytree(output_dir, drive_output_dir, dirs_exist_ok=True)

    with open(summary_path, "r", encoding="utf-8") as handle:
        summary = json.load(handle)

    summary["run_name"] = config["name"]
    summary["repo_output_dir"] = str(output_dir)
    summary["drive_output_dir"] = str(drive_output_dir)
    run_summaries.append(summary)

print("Training finished for all configured runs.")

In [ ]:
summary_rows = []

for summary in run_summaries:
    for model_name, metrics in summary.get("models", {}).items():
        summary_rows.append(
            {
                "run_name": summary["run_name"],
                "model": model_name,
                "n_samples": metrics.get("n_samples"),
                "n_clusters": metrics.get("n_clusters"),
                "silhouette_score": metrics.get("silhouette_score"),
                "davies_bouldin_score": metrics.get("davies_bouldin_score"),
                "calinski_harabasz_score": metrics.get("calinski_harabasz_score"),
                "repo_output_dir": summary["repo_output_dir"],
                "drive_output_dir": summary["drive_output_dir"],
            }
        )

summary_df = pd.DataFrame(summary_rows)
summary_csv = RUN_ROOT / "colab_training_summary.csv"
manifest_json = RUN_ROOT / "colab_training_manifest.json"

summary_df.to_csv(summary_csv, index=False)
with open(manifest_json, "w", encoding="utf-8") as handle:
    json.dump(run_summaries, handle, indent=2)

shutil.copy2(summary_csv, DRIVE_RUN_ROOT / summary_csv.name)
shutil.copy2(manifest_json, DRIVE_RUN_ROOT / manifest_json.name)

summary_df

## Optional GitHub Push

This cell only stages and pushes the generated results after prompting for GitHub username, email, and token. If you skip it, the results still remain in the Colab repo checkout and in Google Drive.

In [ ]:
push_results = (input("Commit and push the new results to GitHub? (y/n) [n]: ").strip().lower() or "n") == "y"

if push_results:
    github_username = prompt_text("GitHub username")
    github_email = prompt_text("GitHub email")
    github_token = prompt_text("GitHub personal access token", secret=True)
    commit_message = prompt_text(
        "Commit message",
        default=f"Add phase boundary Colab results {RUN_TAG}_{TIMESTAMP}",
    )

    relative_results_path = RUN_ROOT.relative_to(repo_root).as_posix()
    notebook_rel_path = Path("Phase Boundary Testing") / "Phase_Boundary_Training_Colab.ipynb"

    run_git(["config", "user.name", github_username], cwd=repo_root)
    run_git(["config", "user.email", github_email], cwd=repo_root)

    add_paths = [relative_results_path]
    if (repo_root / notebook_rel_path).exists():
        add_paths.append(notebook_rel_path.as_posix())

    run_git(["add", *add_paths], cwd=repo_root)

    status = subprocess.run(
        ["git", "status", "--short"],
        cwd=repo_root,
        check=True,
        capture_output=True,
        text=True,
    )
    print(status.stdout)

    if not status.stdout.strip():
        print("No changes to commit.")
    else:
        run_git(["commit", "-m", commit_message], cwd=repo_root)
        branch_name = subprocess.check_output(
            ["git", "branch", "--show-current"],
            cwd=repo_root,
            text=True,
        ).strip()
        run_git(["push", "origin", branch_name], cwd=repo_root, username=github_username, token=github_token)
        print(f"Pushed results to origin/{branch_name}")
else:
    print("Skipping GitHub push.")